In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer, make_column_selector, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import r2_score, make_scorer

import optuna

import xgboost as xgb

from model import Regressor
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

In [2]:
cta_df = pd.read_parquet('../feature_engineer/output/cta_ridership_with_features.parquet')
cta_df = cta_df.reset_index(drop=True)

In [13]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,...,o,location,lat,lon,line,year,month,day,day_of_week_num,day_of_week_name
0,40350,UIC-Halsted,2001-01-01,U,273,40350,False,True,False,False,...,False,"{""latitude"":""41.875474"",""longitude"":""-87.64970...",41.875474,-87.649707,blue,2001,1,1,0,Monday
1,41130,Halsted-Orange,2001-01-01,U,306,41130,False,False,False,False,...,True,"{""latitude"":""41.84678"",""longitude"":""-87.648088...",41.846780,-87.648088,orange,2001,1,1,0,Monday
2,40760,Granville,2001-01-01,U,1059,40760,True,False,False,False,...,False,"{""latitude"":""41.993664"",""longitude"":""-87.65920...",41.993664,-87.659202,red,2001,1,1,0,Monday
3,40070,Jackson/Dearborn,2001-01-01,U,649,40070,False,True,False,False,...,False,"{""latitude"":""41.878183"",""longitude"":""-87.62929...",41.878183,-87.629296,blue,2001,1,1,0,Monday
4,40090,Damen-Brown,2001-01-01,U,411,40090,False,False,False,True,...,False,"{""latitude"":""41.966286"",""longitude"":""-87.67863...",41.966286,-87.678639,brown,2001,1,1,0,Monday
5,40590,Damen/Milwaukee,2001-01-01,U,870,40590,False,True,False,False,...,False,"{""latitude"":""41.909744"",""longitude"":""-87.67743...",41.909744,-87.677437,blue,2001,1,1,0,Monday
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,40720,False,False,True,False,...,False,"{""latitude"":""41.780309"",""longitude"":""-87.60585...",41.780309,-87.605857,green,2001,1,1,0,Monday
7,41260,Austin-Lake,2001-01-01,U,399,41260,False,False,True,False,...,False,"{""latitude"":""41.887293"",""longitude"":""-87.77413...",41.887293,-87.774135,green,2001,1,1,0,Monday
8,40230,Cumberland,2001-01-01,U,788,40230,False,True,False,False,...,False,"{""latitude"":""41.984246"",""longitude"":""-87.83802...",41.984246,-87.838028,blue,2001,1,1,0,Monday
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,41120,False,False,True,False,...,False,"{""latitude"":""41.831677"",""longitude"":""-87.62582...",41.831677,-87.625826,green,2001,1,1,0,Monday


# Pre-processing

## Encode categorical features

In [3]:
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
onehot_encoder = OneHotEncoder()
scaler = StandardScaler()

preprocessor = make_column_transformer(
    # (ordinal_encoder, make_column_selector(dtype_include=object)),
    (onehot_encoder, make_column_selector(dtype_include=object)),
    (scaler, make_column_selector(dtype_include='number')),
    remainder='passthrough'
)

In [4]:
X = preprocessor.fit_transform(cta_df[['line', 'year', 'month', 'day', 'day_of_week_num', 'day_of_week_name', 'lat', 'lon']])
y = cta_df['rides']

# Train-test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
y_train = y_train.astype(float)
y_test = y_test.astype(float)

# Models

Let's do a first pass without any hyperparameter tuning to get a sense of which models have the most predictive power.

## OLS

In [17]:
mod_ols = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
)

mod_ols.fit(X_train, y_train)
y_pred_ols = mod_ols.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_ols))

R squared:  0.13097915308938846


## RF

In [14]:
mod_rf = TransformedTargetRegressor(
    regressor=RandomForestRegressor(n_estimators=10, random_state=42),
    func=np.log1p,
    inverse_func=np.expm1
)

mod_rf.fit(X_train, y_train)
y_pred_rf = mod_rf.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_rf))

R squared:  0.9678180953995276


## XGBoost

In [19]:
mod_xgb = TransformedTargetRegressor(
    regressor=xgb.XGBRegressor(random_state=42),
    func=np.log1p,
    inverse_func=np.expm1
)

mod_xgb.fit(X_train, y_train)
y_pred_xgb = mod_xgb.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_xgb))

R squared:  0.9379040151884488


## Neural network using PyTorch

In [20]:
torch.manual_seed(42)

# Preprocess ----
# Log transform
y_train_log = np.log1p(y_train).values.reshape(-1, 1)
y_test_log = np.log1p(y_test).values.reshape(-1, 1)

y_train_scaled = scaler.fit_transform(y_train_log)
y_test_scaled = scaler.transform(y_test_log)

# Initialize
model = Regressor(n_in=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train_scaled).float()

for epoch in tqdm(range(500)):
    optimizer.zero_grad()
    preds = model(X_train_tensor)
    loss = criterion(preds, y_train_tensor)
    loss.backward()
    optimizer.step()

# Evaluate
model.eval()

with torch.no_grad():
    y_pred_scaled = model(torch.from_numpy(X_test).float()).numpy()
    y_pred = np.expm1(scaler.inverse_transform(y_pred_scaled))
    r2_nn = r2_score(y_test, y_pred)
    print("R squared: ", r2_nn)

100%|██████████| 500/500 [04:11<00:00,  1.98it/s]

R squared:  0.37820722970516496


## Tuned XGB

### Optuna

In [39]:
# Define R-squared scorer with inverse log transform
def r2_scorer_inverse(y_true, y_pred):
    y_true_inverse = np.expm1(y_true)
    y_pred_inverse = np.expm1(y_pred)
    return r2_score(y_true_inverse, y_pred_inverse)

def objective(trial, x, y):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 10)
    }
    
    model_xgb = xgb.XGBRegressor(
        random_state=42, 
        n_estimators=params['n_estimators'], 
        max_depth=params['max_depth'],
        tree_method='hist'
    )
        
    # Use cross-validation with custom scorer
    cv_results = cross_validate(
        model_xgb, x, y, cv=3,
        scoring={'r2_inverse': make_scorer(r2_scorer_inverse)}
    )
    
    return cv_results['test_r2_inverse'].mean()


In [43]:
y_log = np.log1p(y.astype(float))

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: objective(trial, X, y_log), n_trials=10)

[I 2026-02-17 10:28:30,501] A new study created in memory with name: no-name-f7af1e06-c659-4ab5-9e96-5231a6c77c7c
[I 2026-02-17 10:28:45,898] Trial 0 finished with value: 0.5868508442230929 and parameters: {'n_estimators': 177, 'max_depth': 7}. Best is trial 0 with value: 0.5868508442230929.
[I 2026-02-17 10:28:56,660] Trial 1 finished with value: 0.5815990481298782 and parameters: {'n_estimators': 116, 'max_depth': 7}. Best is trial 0 with value: 0.5868508442230929.
[I 2026-02-17 10:29:04,870] Trial 2 finished with value: 0.5649793390104593 and parameters: {'n_estimators': 109, 'max_depth': 5}. Best is trial 0 with value: 0.5868508442230929.
[I 2026-02-17 10:29:20,264] Trial 3 finished with value: 0.5703251463678057 and parameters: {'n_estimators': 155, 'max_depth': 8}. Best is trial 0 with value: 0.5868508442230929.
[I 2026-02-17 10:29:42,021] Trial 4 finished with value: 0.5692091878009049 and parameters: {'n_estimators': 200, 'max_depth': 8}. Best is trial 0 with value: 0.586850844

In [44]:
print(f"Best params is {study.best_params} with value {study.best_value}")

Best params is {'n_estimators': 145, 'max_depth': 6} with value 0.5990121665461164


In [45]:
# Predict using the best set of hyperparameters
mod_xgb_tuned = TransformedTargetRegressor(
    regressor=xgb.XGBRegressor(
        random_state=42, 
        n_estimators=study.best_params['n_estimators'],
        max_depth=study.best_params['max_depth'],
        tree_method='hist'
    ),
    func=np.log1p,
    inverse_func=np.expm1
)

mod_xgb_tuned.fit(X_train, y_train)
y_pred_xgb_tuned = mod_xgb_tuned.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_xgb_tuned))

R squared:  0.9444727422350961
